# BC카드 전체 페이지 카드목록 수집

In [5]:
import requests
import pandas as pd

url = "https://www.bccard.com/app/card/CreditSearch.do"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.bccard.com/app/card/CreditCardMain.do",
}

data = {
    "retKey": "json",
    "pageNo": "1"
}

res = requests.post(url, headers=headers, data=data)
res.raise_for_status()

result = res.json()

print("전체 카드 수:", result.get("TOTAL"))
print("전체 페이지 수:", result.get("PAGE_COUNT"))
print("현재 페이지:", result.get("CURRENT_PAGE"))

cards = result.get("CARDGDS", [])

df_page1 = pd.DataFrame(cards)

display(df_page1[[
    "cardGdsNm",
    "affiFirmNo",
    "mbNo",
    "mainBnftCtnt",
    "CARD_GDS_IMG"
]].head())

전체 카드 수: 158
전체 페이지 수: 16
현재 페이지: 1


,cardGdsNm,affiFirmNo,mbNo,mainBnftCtnt,CARD_GDS_IMG
0,[BNK경남] 경남은행 K-패스 카드,104511,039,대중교통 요금 15% 할인,/images/individual/card/renew/list/card_104511...
1,[Sh수협] 더 아우름 카드,104474,007,프리미엄 바우처 제공,/images/individual/card/renew/list/card_104474...
2,[BC바로] BC 바로 ZONE 카드,104520,050,온라인몰/커피/배달 7% 할인,/images/individual/card/renew/list/card_104520...
3,[BC바로] BC 바로 에어마스터,104452,050,"1,500원당 1마일리지 적립",/images/individual/card/renew/list/card_104452...
4,[BC바로] BC 바로 에어맥스,104453,050,"1,500원당 1마일리지 적립",/images/individual/card/renew/list/card_104453...


In [6]:
import requests
import pandas as pd
import time

url = "https://www.bccard.com/app/card/CreditSearch.do"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.bccard.com/app/card/CreditCardMain.do",
}

all_cards = []

# 1페이지 먼저 요청해서 전체 페이지 수 확인
data = {
    "retKey": "json",
    "pageNo": "1"
}

res = requests.post(url, headers=headers, data=data)
res.raise_for_status()
result = res.json()

total_count = result.get("TOTAL")
page_count = result.get("PAGE_COUNT")

print("전체 카드 수:", total_count)
print("전체 페이지 수:", page_count)

# 전체 페이지 반복 수집
for page_no in range(1, page_count + 1):
    
    data = {
        "retKey": "json",
        "pageNo": str(page_no)
    }
    
    res = requests.post(url, headers=headers, data=data)
    res.raise_for_status()
    
    result = res.json()
    cards = result.get("CARDGDS", [])
    
    print(f"{page_no}페이지 수집 완료: {len(cards)}개")
    
    all_cards.extend(cards)
    
    time.sleep(0.3)

# DataFrame 변환
df_cards = pd.DataFrame(all_cards)

# 상세페이지 URL 만들기
df_cards["detail_url"] = (
    "https://www.bccard.com/app/card/CreditCardMain.do?"
    + "gdsno=" + df_cards["affiFirmNo"].astype(str)
    + "&mbkNo=" + df_cards["mbNo"].astype(str)
)

# 이미지 URL 만들기
df_cards["image_url"] = "https://www.bccard.com" + df_cards["CARD_GDS_IMG"].astype(str)

# 필요한 컬럼만 정리
df_cards_final = df_cards[[
    "cardGdsNm",
    "affiFirmNo",
    "cardGdsNo",
    "mbNo",
    "mainBnftCtnt",
    "mainBnftCtnt2",
    "mainBnftCtnt3",
    "detail_url",
    "image_url",
    "bcIssuClss",
    "mcIssuClss",
    "visaIssuClss",
    "amexIssuClss",
    "jcbIssuClss",
    "cupIssuClss"
]].copy()

# 중복 제거
df_cards_final = df_cards_final.drop_duplicates(subset=["affiFirmNo", "mbNo"])

print("최종 수집 카드 수:", len(df_cards_final))

display(df_cards_final.head())

# CSV 저장
df_cards_final.to_csv("bc_card_list_all.csv", index=False, encoding="utf-8-sig")

전체 카드 수: 158
전체 페이지 수: 16
1페이지 수집 완료: 10개
2페이지 수집 완료: 10개
3페이지 수집 완료: 10개
4페이지 수집 완료: 10개
5페이지 수집 완료: 10개
6페이지 수집 완료: 10개
7페이지 수집 완료: 10개
8페이지 수집 완료: 10개
9페이지 수집 완료: 10개
10페이지 수집 완료: 10개
11페이지 수집 완료: 10개
12페이지 수집 완료: 10개
13페이지 수집 완료: 10개
14페이지 수집 완료: 10개
15페이지 수집 완료: 10개
16페이지 수집 완료: 8개
최종 수집 카드 수: 158


,cardGdsNm,affiFirmNo,cardGdsNo,mbNo,mainBnftCtnt,mainBnftCtnt2,mainBnftCtnt3,detail_url,image_url,bcIssuClss,mcIssuClss,visaIssuClss,amexIssuClss,jcbIssuClss,cupIssuClss
0,[BNK경남] 경남은행 K-패스 카드,104511,1045110000,039,대중교통 요금 15% 할인,,,https://www.bccard.com/app/card/CreditCardMain...,https://www.bccard.com/images/individual/card/...,1,1,0,0,0,0
1,[Sh수협] 더 아우름 카드,104474,1044740000,007,프리미엄 바우처 제공,,,https://www.bccard.com/app/card/CreditCardMain...,https://www.bccard.com/images/individual/card/...,1,1,0,0,0,0
2,[BC바로] BC 바로 ZONE 카드,104520,1045200000,050,온라인몰/커피/배달 7% 할인,,,https://www.bccard.com/app/card/CreditCardMain...,https://www.bccard.com/images/individual/card/...,1,1,0,0,0,0
3,[BC바로] BC 바로 에어마스터,104452,1044520000,050,"1,500원당 1마일리지 적립",,,https://www.bccard.com/app/card/CreditCardMain...,https://www.bccard.com/images/individual/card/...,1,1,0,0,0,0
4,[BC바로] BC 바로 에어맥스,104453,1044530000,050,"1,500원당 1마일리지 적립",,,https://www.bccard.com/app/card/CreditCardMain...,https://www.bccard.com/images/individual/card/...,1,0,0,1,0,0


# 카드 상세 혜택 페이지 수집

In [12]:
import requests
from bs4 import BeautifulSoup

def parse_card_detail(detail_url):
    
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.bccard.com/app/card/CreditCardMain.do",
    }
    
    res = requests.get(detail_url, headers=headers)
    res.raise_for_status()
    
    soup = BeautifulSoup(res.text, "html.parser")
    
    # 혜택
    benefit_items = soup.select("#divCardDetail_info .cardViewBody .item")
    
    results = []
    
    for idx, item in enumerate(benefit_items, start=1):
        
        title_tag = item.select_one(".tit p")
        benefit_title = title_tag.get_text(" ", strip=True) if title_tag else ""
        
        view = item.select_one(".view")
        if not view:
            continue
        
        sub_titles = [
            p.get_text(" ", strip=True)
            for p in view.select("p.sub_tit")
            if p.get_text(" ", strip=True)
        ]
        
        list_texts = [
            li.get_text(" ", strip=True)
            for li in view.select("ul.txt_list li")
            if li.get_text(" ", strip=True)
        ]
        
        notes = [
            p.get_text(" ", strip=True)
            for p in view.select("p.txt_mark")
            if p.get_text(" ", strip=True)
        ]
        
        tables = []
        
        for table in view.select("table"):
            rows = []
            
            for tr in table.select("tr"):
                cells = [
                    cell.get_text(" ", strip=True)
                    for cell in tr.select("th, td")
                ]
                
                if cells:
                    rows.append(cells)
            
            if rows:
                tables.append(rows)
        
        full_text = view.get_text("\n", strip=True)
        
        results.append({
            "benefit_order": idx,
            "benefit_title": benefit_title,
            "sub_titles": sub_titles,
            "list_texts": list_texts,
            "notes": notes,
            "tables": tables,
            "full_text": full_text
        })
    
    return results

In [13]:
test_url = df_cards_final.loc[2, "detail_url"]

detail_results = parse_card_detail(test_url)

print("수집된 혜택 개수:", len(detail_results))

for r in detail_results:
    print("=" * 80)
    print("혜택명:", r["benefit_title"])
    print("소제목:", r["sub_titles"])
    print("리스트:", r["list_texts"][:3])
    print("주의:", r["notes"][:2])
    print("표 개수:", len(r["tables"]))
    print("전체 텍스트 앞부분:")
    print(r["full_text"][:500])

수집된 혜택 개수: 7
혜택명: EAT ZONE
소제목: []
리스트: ['커피, 간식 : 현장결제 또는 각 브랜드의 공식앱을 통한 온라인 결제 건에 한해 할인 적용 (백화점, 면세점, 할인점, 공항, 기차역 등 임대 매장 제외)', '배달 : 공식 홈페이지 및 앱을 통한 결제 건에 한해 할인 적용 (가맹점 및 배달원 직접 결제 건 제외)']
주의: []
표 개수: 1
전체 텍스트 앞부분:
카테고리별 가맹점, 월 할인 한도 안내표
카테고리
대상 가맹점
월 할인 한도
전월실적
30만원 이상
전월실적
60만원 이상
커피
스타벅스, 투썸플레이스, 이디야커피, 메가MGC커피, 컴포즈커피
5천원
1만원
간식
맥도날드, 버거킹, 롯데리아, 맘스터치, 써브웨이
배달
배달의민족, 쿠팡이츠
커피, 간식 : 현장결제 또는 각 브랜드의 공식앱을 통한 온라인 결제 건에 한해 할인 적용 (백화점, 면세점, 할인점, 공항, 기차역 등 임대 매장 제외)
배달 : 공식 홈페이지 및 앱을 통한 결제 건에 한해 할인 적용 (가맹점 및 배달원 직접 결제 건 제외)
혜택명: LIFE ZONE
소제목: []
리스트: ['온라인몰 : 대상 가맹점 공식 홈페이지 또는 모바일앱 이용 시 할인 적용 (대상 가맹점에서 각종 간편결제를 통한 결제 시 할인 제외될 수 있음)', 'LIFE-ZONE 내 ‘쿠팡’은 온라인 쇼핑몰(쿠팡이츠 제외) 결제 건에 한해 할인 적용 (‘쿠팡이츠’ 결제는 EAT-ZONE 할인 적용, EAT/LIFE 중복 적용 불가)', '오프라인샵 : 오프라인 매장 결제 건에 한해 할인 적용 (백화점, 할인점, 공항, 기차역 등 임대 매장 제외)']
주의: []
표 개수: 1
전체 텍스트 앞부분:
카테고리별 가맹점, 월 할인 한도 안내표
카테고리
대상 가맹점
월 할인 한도
전월실적
30만원 이상
전월실적
60만원 이상
온라인몰
쿠팡, 무신사, 지그재그, 오늘의집, 크림
5천원
1만원
오프라인샵
다이소, 올리브영
온라인몰 : 대상 가맹점 공식 홈페이지 또는 모바일앱 이용 시 할인 적용
(대상 

In [14]:
import time
import pandas as pd

all_detail_rows = []

# 먼저 10개만 테스트
df_test = df_cards_final.head(10)

for idx, row in df_test.iterrows():
    
    card_name = row["cardGdsNm"]
    detail_url = row["detail_url"]
    
    print(f"[{idx+1}/{len(df_test)}] 수집중: {card_name}")
    
    try:
        detail_results = parse_card_detail(detail_url)
        
        print(f"  → 혜택 {len(detail_results)}개 수집")
        
        for detail in detail_results:
            all_detail_rows.append({
                "card_name": card_name,
                "detail_url": detail_url,
                "benefit_order": detail["benefit_order"],
                "benefit_title": detail["benefit_title"],
                "sub_titles": " | ".join(detail["sub_titles"]),
                "list_texts": " | ".join(detail["list_texts"]),
                "notes": " | ".join(detail["notes"]),
                "full_text": detail["full_text"]
            })
            
    except Exception as e:
        print("  → 에러:", e)
    
    time.sleep(0.5)

df_detail_test = pd.DataFrame(all_detail_rows)

print("총 상세 혜택 수:", len(df_detail_test))

display(df_detail_test.head(20))

df_detail_test.to_csv(
    "bc_card_detail_test_10.csv",
    index=False,
    encoding="utf-8-sig"
)

[1/10] 수집중: [BNK경남] 경남은행 K-패스 카드
  → 혜택 3개 수집
[2/10] 수집중: [Sh수협] 더 아우름 카드
  → 혜택 5개 수집
[3/10] 수집중: [BC바로] BC 바로 ZONE 카드
  → 혜택 7개 수집
[4/10] 수집중: [BC바로] BC 바로 에어마스터
  → 혜택 5개 수집
[5/10] 수집중: [BC바로] BC 바로 에어맥스
  → 혜택 5개 수집
[6/10] 수집중: [광주] 대한항공 SKYPASS 카드(일반형)
  → 혜택 6개 수집
[7/10] 수집중: [광주] 대한항공 SKYPASS 카드(플래티늄형)
  → 혜택 7개 수집
[8/10] 수집중: [BNK경남] REXⅡ 카드
  → 혜택 4개 수집
[9/10] 수집중: [iM뱅크] iM 트래블 카드
  → 혜택 4개 수집
[10/10] 수집중: [광주] 기아챔피언스카드
  → 혜택 4개 수집
총 상세 혜택 수: 50


,card_name,detail_url,benefit_order,benefit_title,sub_titles,list_texts,notes,full_text
0,[BNK경남] 경남은행 K-패스 카드,https://www.bccard.com/app/card/CreditCardMain...,1,모빌리티 할인 서비스(결제일 차감 청구),대중교통 요금 15% 할인 | 친환경 모빌리티 10% 할인 | [모빌리티 할인 서비...,"대상 가맹점 : 버스(시내버스, 마을버스, 광역버스)지하철 및 GTX - 시외버스 ...",,"대중교통 요금 15% 할인\n대상 가맹점 : 버스(시내버스, 마을버스, 광역버스)지..."
1,[BNK경남] 경남은행 K-패스 카드,https://www.bccard.com/app/card/CreditCardMain...,2,생활 할인 서비스(결제일 차감 청구),"이동통신요금 5% 할인 (월 1회, 최대 1,500원 할인) | 온라인 쇼핑/배달앱...","대상 가맹점 : SKT, KT, LGU+ - 이동통신요금을 본 카드로 대상 통신사에...","※ 동물병원 할인 제외 | ※ 백화점, 할인점, 쇼핑몰 등의 임대매장 제외 | ※ ...","이동통신요금 5% 할인 (월 1회, 최대 1,500원 할인)\n대상 가맹점 : SK..."
2,[BNK경남] 경남은행 K-패스 카드,https://www.bccard.com/app/card/CreditCardMain...,3,기타 안내,카드발급안내 | 해외 이용 안내 | 연회비 청구 및 반환 기준 | 고객 상담 안내,해외 이용시(해외 웹사이트 포함) 이용금액에 국제브랜드사가 부과하는 수수료 (Mas...,※ BC카드 대외결제은행은 BC카드 홈페이지(고객센터-해외이용안내)에서 확인가능합니...,카드발급안내\n카드발급안내\n구분\n내용\n발급대상\n개인회원(가족회원 불가)\n브...
3,[Sh수협] 더 아우름 카드,https://www.bccard.com/app/card/CreditCardMain...,1,프리미엄 바우처 (Premium Voucher),바우처 4가지 중 1가지를 선택하여 이용(연1회) | 바우처 신청채널 | 바우처 신...,"BC카드 고객센터 ☏1566-7890 (평일 09:00 ~ 18:00), BC페이북...","※ 실적제외업종 : 할인받은 이용금액 전체, 장기카드대출(카드론), 단기카드대출(현...",바우처 4가지 중 1가지를 선택하여 이용(연1회)\n프리미엄 바우처 안내\nHote...
4,[Sh수협] 더 아우름 카드,https://www.bccard.com/app/card/CreditCardMain...,2,할인서비스,라이프스타일 패키지(옵션 패키지 중 택1) | 할인 서비스 공통기준 | 할인 제외 ...,서비스 안내 : 라이프스타일에 따른 옵션 패키지 선택 시 해당 패키지 업종 5% 결...,※ 모든 서비스에 적용되는 가맹점 및 업종은 BC카드에 등록된 가맹점 정보를 기준으...,라이프스타일 패키지(옵션 패키지 중 택1)\n서비스 안내 : 라이프스타일에 따른 옵...
5,[Sh수협] 더 아우름 카드,https://www.bccard.com/app/card/CreditCardMain...,3,마스터카드 월드서비스(공통),2026 Mastercard World 공통서비스 | [HOTEL] 호텔 서비스 |...,국내특급호텔(약42여개)F&B 5 ~ 15%할인 | 국내특급호텔(약44여개) 객실/...,※ 발급 받으신 카드는 Mastercard WORLD 등급의 국제 브랜드 서비스가 ...,2026 Mastercard World 공통서비스\n※ 발급 받으신 카드는 Mast...
6,[Sh수협] 더 아우름 카드,https://www.bccard.com/app/card/CreditCardMain...,4,마스터카드 월드서비스(선택),2025 Mastercard World 선택서비스 | Mastercard 프리미엄 ...,전월실적(이용시점 기준 전월 1일 ~ 말일) 50만원 이상 사용고객 | 전월실적 제...,※ Mastercard 프리미엄 서비스의 상세한 내용은 Mastercard홈페이지 ...,2025 Mastercard World 선택서비스\n※ Mastercard 프리미엄...
7,[Sh수협] 더 아우름 카드,https://www.bccard.com/app/card/CreditCardMain...,5,기타 안내,카드발급안내 | 연회비 안내 | 해외이용수수료 안내 | [해외 이용시 청구금액 산출...,국내외겸용(Mastercard) 발급 시 Mastercard World 등급의 서비...,※ 연회비는 카드발급 시점을 기준으로 1년 단위로 청구되며 초회년도 연회비 면제조건...,카드발급안내\n국내외겸용(Mastercard) 발급 시 Mastercard Worl...
8,[BC바로] BC 바로 ZONE 카드,https://www.bccard.com/app/card/CreditCardMain...,1,EAT ZONE,,"커피, 간식 : 현장결제 또는 각 브랜드의 공식앱을 통한 온라인 결제 건에 한해 할...",,"카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
9,[BC바로] BC 바로 ZONE 카드,https://www.bccard.com/app/card/CreditCardMain...,2,LIFE ZONE,,온라인몰 : 대상 가맹점 공식 홈페이지 또는 모바일앱 이용 시 할인 적용 (대상 가...,,"카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."


#### 158개 카드 전체 수집

In [15]:
import time
import pandas as pd

all_detail_rows = []

for i, row in df_cards_final.iterrows():
    
    card_name = row["cardGdsNm"]
    detail_url = row["detail_url"]
    
    print(f"[{i+1}/{len(df_cards_final)}] 수집중: {card_name}")
    
    try:
        detail_results = parse_card_detail(detail_url)
        
        print(f"  → 혜택 {len(detail_results)}개 수집")
        
        for detail in detail_results:
            all_detail_rows.append({
                "card_name": card_name,
                "affiFirmNo": row["affiFirmNo"],
                "mbNo": row["mbNo"],
                "main_benefit": row["mainBnftCtnt"],
                "detail_url": detail_url,
                "benefit_order": detail["benefit_order"],
                "benefit_title": detail["benefit_title"],
                "sub_titles": " | ".join(detail["sub_titles"]),
                "list_texts": " | ".join(detail["list_texts"]),
                "notes": " | ".join(detail["notes"]),
                "full_text": detail["full_text"]
            })
            
    except Exception as e:
        print(f"  → 에러 발생: {e}")
    
    time.sleep(0.5)

df_detail_all = pd.DataFrame(all_detail_rows)

print("총 상세 혜택 수:", len(df_detail_all))

display(df_detail_all.head(20))

df_detail_all.to_csv(
    "bc_card_detail_all.csv",
    index=False,
    encoding="utf-8-sig"
)

[1/158] 수집중: [BNK경남] 경남은행 K-패스 카드
  → 혜택 3개 수집
[2/158] 수집중: [Sh수협] 더 아우름 카드
  → 혜택 5개 수집
[3/158] 수집중: [BC바로] BC 바로 ZONE 카드
  → 혜택 7개 수집
[4/158] 수집중: [BC바로] BC 바로 에어마스터
  → 혜택 5개 수집
[5/158] 수집중: [BC바로] BC 바로 에어맥스
  → 혜택 5개 수집
[6/158] 수집중: [광주] 대한항공 SKYPASS 카드(일반형)
  → 혜택 6개 수집
[7/158] 수집중: [광주] 대한항공 SKYPASS 카드(플래티늄형)
  → 혜택 7개 수집
[8/158] 수집중: [BNK경남] REXⅡ 카드
  → 혜택 4개 수집
[9/158] 수집중: [iM뱅크] iM 트래블 카드
  → 혜택 4개 수집
[10/158] 수집중: [광주] 기아챔피언스카드
  → 혜택 4개 수집
[11/158] 수집중: [Sh수협] All드림카드
  → 혜택 9개 수집
[12/158] 수집중: [BC바로] KT SUPER DC BC 바로카드
  → 혜택 3개 수집
[13/158] 수집중: [IBK기업은행] IBK포인트3.8
  → 혜택 1개 수집
[14/158] 수집중: [BC바로] BC 바로 기후동행카드
  → 혜택 7개 수집
[15/158] 수집중: [BNK부산] REXⅡ 카드
  → 혜택 5개 수집
[16/158] 수집중: [IBK기업은행] IBK포인트
  → 혜택 1개 수집
[17/158] 수집중: [BC바로] MACAO 카드
  → 혜택 4개 수집
[18/158] 수집중: [광주] 오일모아카드
  → 혜택 4개 수집
[19/158] 수집중: [광주] 에듀플러스카드
  → 혜택 4개 수집
[20/158] 수집중: [BC바로] KaPick
  → 혜택 7개 수집
[21/158] 수집중: [BC바로] KT 마이알뜰폰 BC 바로카드
  → 혜택 3개 수집
[22/158] 수집중: [BNK경남] 키스해링 신용카드
  → 혜택 1개 수집
[23/158

,card_name,affiFirmNo,mbNo,main_benefit,detail_url,benefit_order,benefit_title,sub_titles,list_texts,notes,full_text
0,[BNK경남] 경남은행 K-패스 카드,104511,039,대중교통 요금 15% 할인,https://www.bccard.com/app/card/CreditCardMain...,1,모빌리티 할인 서비스(결제일 차감 청구),대중교통 요금 15% 할인 | 친환경 모빌리티 10% 할인 | [모빌리티 할인 서비...,"대상 가맹점 : 버스(시내버스, 마을버스, 광역버스)지하철 및 GTX - 시외버스 ...",,"대중교통 요금 15% 할인\n대상 가맹점 : 버스(시내버스, 마을버스, 광역버스)지..."
1,[BNK경남] 경남은행 K-패스 카드,104511,039,대중교통 요금 15% 할인,https://www.bccard.com/app/card/CreditCardMain...,2,생활 할인 서비스(결제일 차감 청구),"이동통신요금 5% 할인 (월 1회, 최대 1,500원 할인) | 온라인 쇼핑/배달앱...","대상 가맹점 : SKT, KT, LGU+ - 이동통신요금을 본 카드로 대상 통신사에...","※ 동물병원 할인 제외 | ※ 백화점, 할인점, 쇼핑몰 등의 임대매장 제외 | ※ ...","이동통신요금 5% 할인 (월 1회, 최대 1,500원 할인)\n대상 가맹점 : SK..."
2,[BNK경남] 경남은행 K-패스 카드,104511,039,대중교통 요금 15% 할인,https://www.bccard.com/app/card/CreditCardMain...,3,기타 안내,카드발급안내 | 해외 이용 안내 | 연회비 청구 및 반환 기준 | 고객 상담 안내,해외 이용시(해외 웹사이트 포함) 이용금액에 국제브랜드사가 부과하는 수수료 (Mas...,※ BC카드 대외결제은행은 BC카드 홈페이지(고객센터-해외이용안내)에서 확인가능합니...,카드발급안내\n카드발급안내\n구분\n내용\n발급대상\n개인회원(가족회원 불가)\n브...
3,[Sh수협] 더 아우름 카드,104474,007,프리미엄 바우처 제공,https://www.bccard.com/app/card/CreditCardMain...,1,프리미엄 바우처 (Premium Voucher),바우처 4가지 중 1가지를 선택하여 이용(연1회) | 바우처 신청채널 | 바우처 신...,"BC카드 고객센터 ☏1566-7890 (평일 09:00 ~ 18:00), BC페이북...","※ 실적제외업종 : 할인받은 이용금액 전체, 장기카드대출(카드론), 단기카드대출(현...",바우처 4가지 중 1가지를 선택하여 이용(연1회)\n프리미엄 바우처 안내\nHote...
4,[Sh수협] 더 아우름 카드,104474,007,프리미엄 바우처 제공,https://www.bccard.com/app/card/CreditCardMain...,2,할인서비스,라이프스타일 패키지(옵션 패키지 중 택1) | 할인 서비스 공통기준 | 할인 제외 ...,서비스 안내 : 라이프스타일에 따른 옵션 패키지 선택 시 해당 패키지 업종 5% 결...,※ 모든 서비스에 적용되는 가맹점 및 업종은 BC카드에 등록된 가맹점 정보를 기준으...,라이프스타일 패키지(옵션 패키지 중 택1)\n서비스 안내 : 라이프스타일에 따른 옵...
5,[Sh수협] 더 아우름 카드,104474,007,프리미엄 바우처 제공,https://www.bccard.com/app/card/CreditCardMain...,3,마스터카드 월드서비스(공통),2026 Mastercard World 공통서비스 | [HOTEL] 호텔 서비스 |...,국내특급호텔(약42여개)F&B 5 ~ 15%할인 | 국내특급호텔(약44여개) 객실/...,※ 발급 받으신 카드는 Mastercard WORLD 등급의 국제 브랜드 서비스가 ...,2026 Mastercard World 공통서비스\n※ 발급 받으신 카드는 Mast...
6,[Sh수협] 더 아우름 카드,104474,007,프리미엄 바우처 제공,https://www.bccard.com/app/card/CreditCardMain...,4,마스터카드 월드서비스(선택),2025 Mastercard World 선택서비스 | Mastercard 프리미엄 ...,전월실적(이용시점 기준 전월 1일 ~ 말일) 50만원 이상 사용고객 | 전월실적 제...,※ Mastercard 프리미엄 서비스의 상세한 내용은 Mastercard홈페이지 ...,2025 Mastercard World 선택서비스\n※ Mastercard 프리미엄...
7,[Sh수협] 더 아우름 카드,104474,007,프리미엄 바우처 제공,https://www.bccard.com/app/card/CreditCardMain...,5,기타 안내,카드발급안내 | 연회비 안내 | 해외이용수수료 안내 | [해외 이용시 청구금액 산출...,국내외겸용(Mastercard) 발급 시 Mastercard World 등급의 서비...,※ 연회비는 카드발급 시점을 기준으로 1년 단위로 청구되며 초회년도 연회비 면제조건...,카드발급안내\n국내외겸용(Mastercard) 발급 시 Mastercard Worl...
8,[BC바로] BC 바로 ZONE 카드,104520,050,온라인몰/커피/배달 7% 할인,https://www.bccard.com/app/card/CreditCardMain...,1,EAT ZONE,,"커피, 간식 : 현장결제 또는 각 브랜드의 공식앱을 통한 온라인 결제 건에 한해 할...",,"카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
9,[BC바로] BC 바로 ZONE 카드,104520,050,온라인몰/커피/배달 7% 할인,https://www.bccard.com/app/card/CreditCardMain...,2,LIFE ZONE,,온라인몰 : 대상 가맹점 공식 홈페이지 또는 모바일앱 이용 시 할인 적용 (대상 가...,,"카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."


In [16]:
df_detail_all[[
    "card_name",
    "benefit_order",
    "benefit_title",
    "full_text"
]].head(30)

,card_name,benefit_order,benefit_title,full_text
0,[BNK경남] 경남은행 K-패스 카드,1,모빌리티 할인 서비스(결제일 차감 청구),"대중교통 요금 15% 할인\n대상 가맹점 : 버스(시내버스, 마을버스, 광역버스)지..."
1,[BNK경남] 경남은행 K-패스 카드,2,생활 할인 서비스(결제일 차감 청구),"이동통신요금 5% 할인 (월 1회, 최대 1,500원 할인)\n대상 가맹점 : SK..."
2,[BNK경남] 경남은행 K-패스 카드,3,기타 안내,카드발급안내\n카드발급안내\n구분\n내용\n발급대상\n개인회원(가족회원 불가)\n브...
3,[Sh수협] 더 아우름 카드,1,프리미엄 바우처 (Premium Voucher),바우처 4가지 중 1가지를 선택하여 이용(연1회)\n프리미엄 바우처 안내\nHote...
4,[Sh수협] 더 아우름 카드,2,할인서비스,라이프스타일 패키지(옵션 패키지 중 택1)\n서비스 안내 : 라이프스타일에 따른 옵...
5,[Sh수협] 더 아우름 카드,3,마스터카드 월드서비스(공통),2026 Mastercard World 공통서비스\n※ 발급 받으신 카드는 Mast...
6,[Sh수협] 더 아우름 카드,4,마스터카드 월드서비스(선택),2025 Mastercard World 선택서비스\n※ Mastercard 프리미엄...
7,[Sh수협] 더 아우름 카드,5,기타 안내,카드발급안내\n국내외겸용(Mastercard) 발급 시 Mastercard Worl...
8,[BC바로] BC 바로 ZONE 카드,1,EAT ZONE,"카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
9,[BC바로] BC 바로 ZONE 카드,2,LIFE ZONE,"카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."


In [17]:
df_detail_all.shape
df_detail_all["card_name"].nunique()

139

In [18]:
# 1) 목록에는 있는데 상세 수집 결과에는 없는 카드 확인
expected_cards = set(df_cards_final["cardGdsNm"])
collected_cards = set(df_detail_all["card_name"])

missing_cards = expected_cards - collected_cards

print("전체 카드 수:", len(expected_cards))
print("상세 수집된 카드 수:", len(collected_cards))
print("누락 카드 수:", len(missing_cards))

df_missing = df_cards_final[df_cards_final["cardGdsNm"].isin(missing_cards)].copy()

display(df_missing[[
    "cardGdsNm",
    "affiFirmNo",
    "mbNo",
    "mainBnftCtnt",
    "detail_url"
]])

전체 카드 수: 157
상세 수집된 카드 수: 139
누락 카드 수: 18


,cardGdsNm,affiFirmNo,mbNo,mainBnftCtnt,detail_url
140,[BC공통] 비씨 다이아몬드 골프야드 카드,242266,999,다이아몬드 등급 VIP 서비스,https://www.bccard.com/app/card/CreditCardMain...
141,[BC공통] 비씨 다이아몬드 TOP 카드,242237,999,다이아몬드 등급 VIP 서비스,https://www.bccard.com/app/card/CreditCardMain...
142,[BC공통] 비씨 다이아몬드 스카이패스 카드,242241,999,다이아몬드 등급 VIP 서비스,https://www.bccard.com/app/card/CreditCardMain...
143,[BC공통] 비씨TOP포인트카드,000000,999,TOP포인트 최대 0.3% 적립,https://www.bccard.com/app/card/CreditCardMain...
144,[BC공통] 독도지킴이카드,221009,999,0.1% 독도지킴이 기금 제공,https://www.bccard.com/app/card/CreditCardMain...
145,[BC공통] 인피니트 TOP 카드,225005,999,인피니트 등급 VIP 서비스,https://www.bccard.com/app/card/CreditCardMain...
146,[BC공통] 인피니트 아시아나클럽 카드,225021,999,인피니트 등급 VIP 서비스,https://www.bccard.com/app/card/CreditCardMain...
147,[BC공통] 인피니트 스카이패스 카드,225018,999,인피니트 등급 VIP 서비스,https://www.bccard.com/app/card/CreditCardMain...
148,[BC공통] BC Miles카드,222804,999,1천원당 최대 2BC Mile 적립,https://www.bccard.com/app/card/CreditCardMain...
149,[BC공통] CarPlus카드,602001,999,CarPlus 0.8% 포인트 적립,https://www.bccard.com/app/card/CreditCardMain...


#### -- 누락카드 18개: 발급 중단 카드 -- 

# 계산용 컬럼으로 정리

In [19]:
df_benefit = df_detail_all.copy()

df_benefit["raw_text"] = (
    df_benefit["benefit_title"].fillna("") + "\n" +
    df_benefit["sub_titles"].fillna("") + "\n" +
    df_benefit["list_texts"].fillna("") + "\n" +
    df_benefit["notes"].fillna("") + "\n" +
    df_benefit["full_text"].fillna("")
)

df_benefit[["card_name", "benefit_title", "raw_text"]].head()

,card_name,benefit_title,raw_text
0,[BNK경남] 경남은행 K-패스 카드,모빌리티 할인 서비스(결제일 차감 청구),모빌리티 할인 서비스(결제일 차감 청구)\n대중교통 요금 15% 할인 | 친환경 모...
1,[BNK경남] 경남은행 K-패스 카드,생활 할인 서비스(결제일 차감 청구),"생활 할인 서비스(결제일 차감 청구)\n이동통신요금 5% 할인 (월 1회, 최대 1..."
2,[BNK경남] 경남은행 K-패스 카드,기타 안내,기타 안내\n카드발급안내 | 해외 이용 안내 | 연회비 청구 및 반환 기준 | 고객...
3,[Sh수협] 더 아우름 카드,프리미엄 바우처 (Premium Voucher),프리미엄 바우처 (Premium Voucher)\n바우처 4가지 중 1가지를 선택하...
4,[Sh수협] 더 아우름 카드,할인서비스,할인서비스\n라이프스타일 패키지(옵션 패키지 중 택1) | 할인 서비스 공통기준 |...


In [20]:
import re
import pandas as pd

def extract_rate(text):
    if pd.isna(text):
        return None
    
    patterns = [
        r"(\d+(?:\.\d+)?)\s*%",
        r"(\d+(?:\.\d+)?)\s*퍼센트"
    ]
    
    for p in patterns:
        m = re.search(p, text)
        if m:
            return float(m.group(1))
    
    return None


def extract_previous_month_spend(text):
    if pd.isna(text):
        return None
    
    patterns = [
        r"전월\s*(?:이용|사용|실적).*?(\d+)\s*만\s*원\s*이상",
        r"(\d+)\s*만\s*원\s*이상.*?전월\s*(?:이용|사용|실적)",
        r"전월.*?(\d+)\s*만원\s*이상"
    ]
    
    for p in patterns:
        m = re.search(p, text)
        if m:
            return int(m.group(1)) * 10000
    
    return None


def extract_monthly_limit(text):
    if pd.isna(text):
        return None
    
    patterns = [
        r"월\s*(?:통합)?\s*(?:할인|적립)?\s*한도\s*[:：]?\s*(\d+(?:,\d+)*)\s*원",
        r"월\s*(\d+(?:,\d+)*)\s*원\s*(?:까지|한도)",
        r"월\s*(\d+)\s*만\s*원\s*(?:까지|한도)",
        r"월\s*최대\s*(\d+(?:,\d+)*)\s*원",
        r"최대\s*(\d+(?:,\d+)*)\s*원\s*(?:할인|적립|캐시백)"
    ]
    
    for p in patterns:
        m = re.search(p, text)
        if m:
            value = m.group(1).replace(",", "")
            return int(float(value))
    
    return None


def classify_benefit_type(text):
    if pd.isna(text):
        return None
    
    if "마일리지" in text:
        return "마일리지"
    elif "캐시백" in text:
        return "캐시백"
    elif "적립" in text:
        return "적립"
    elif "할인" in text:
        return "할인"
    elif "바우처" in text:
        return "바우처"
    else:
        return "기타"


df_benefit["benefit_rate"] = df_benefit["raw_text"].apply(extract_rate)
df_benefit["previous_month_spend"] = df_benefit["raw_text"].apply(extract_previous_month_spend)
df_benefit["monthly_limit"] = df_benefit["raw_text"].apply(extract_monthly_limit)
df_benefit["benefit_type"] = df_benefit["raw_text"].apply(classify_benefit_type)

df_benefit[[
    "card_name",
    "benefit_title",
    "benefit_type",
    "benefit_rate",
    "previous_month_spend",
    "monthly_limit"
]].head(30)

,card_name,benefit_title,benefit_type,benefit_rate,previous_month_spend,monthly_limit
0,[BNK경남] 경남은행 K-패스 카드,모빌리티 할인 서비스(결제일 차감 청구),할인,15.0,NaN,NaN
1,[BNK경남] 경남은행 K-패스 카드,생활 할인 서비스(결제일 차감 청구),할인,5.0,NaN,1500.0
2,[BNK경남] 경남은행 K-패스 카드,기타 안내,기타,1.0,NaN,NaN
3,[Sh수협] 더 아우름 카드,프리미엄 바우처 (Premium Voucher),할인,NaN,NaN,NaN
4,[Sh수협] 더 아우름 카드,할인서비스,할인,5.0,500000.0,5000.0
5,[Sh수협] 더 아우름 카드,마스터카드 월드서비스(공통),할인,15.0,NaN,NaN
6,[Sh수협] 더 아우름 카드,마스터카드 월드서비스(선택),할인,NaN,500000.0,NaN
7,[Sh수협] 더 아우름 카드,기타 안내,바우처,1.0,NaN,NaN
8,[BC바로] BC 바로 ZONE 카드,EAT ZONE,할인,NaN,NaN,NaN
9,[BC바로] BC 바로 ZONE 카드,LIFE ZONE,할인,NaN,NaN,NaN


In [21]:
# BC 바로 ZONE 카드만 확인
zone = df_detail_all[df_detail_all["card_name"].str.contains("BC 바로 ZONE", na=False)]

display(zone[[
    "card_name",
    "benefit_title",
    "full_text"
]])

,card_name,benefit_title,full_text
8,[BC바로] BC 바로 ZONE 카드,EAT ZONE,"카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
9,[BC바로] BC 바로 ZONE 카드,LIFE ZONE,"카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
10,[BC바로] BC 바로 ZONE 카드,PLAY ZONE,"카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
11,[BC바로] BC 바로 ZONE 카드,& ZONE,"카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
12,[BC바로] BC 바로 ZONE 카드,해외 ZONE,"해외가맹점 및 해외 직접 구매 이용시\n전월 실적, 한도 없이 1% 결제일 할인\n..."
13,[BC바로] BC 바로 ZONE 카드,무이자할부,국내 대상 가맹점 2~3개월 상시 무이자 할부\n무이자할부 이용금액은 할인 혜택 제...
14,[BC바로] BC 바로 ZONE 카드,국제 브랜드 서비스,해외겸용 카드의 경우 Mastercard Platinum 등급 기본 서비스만 제공되...


In [22]:
for i, row in zone.iterrows():
    print("=" * 80)
    print(row["benefit_title"])
    print(row["full_text"][:1500])

EAT ZONE
카테고리별 가맹점, 월 할인 한도 안내표
카테고리
대상 가맹점
월 할인 한도
전월실적
30만원 이상
전월실적
60만원 이상
커피
스타벅스, 투썸플레이스, 이디야커피, 메가MGC커피, 컴포즈커피
5천원
1만원
간식
맥도날드, 버거킹, 롯데리아, 맘스터치, 써브웨이
배달
배달의민족, 쿠팡이츠
커피, 간식 : 현장결제 또는 각 브랜드의 공식앱을 통한 온라인 결제 건에 한해 할인 적용 (백화점, 면세점, 할인점, 공항, 기차역 등 임대 매장 제외)
배달 : 공식 홈페이지 및 앱을 통한 결제 건에 한해 할인 적용 (가맹점 및 배달원 직접 결제 건 제외)
LIFE ZONE
카테고리별 가맹점, 월 할인 한도 안내표
카테고리
대상 가맹점
월 할인 한도
전월실적
30만원 이상
전월실적
60만원 이상
온라인몰
쿠팡, 무신사, 지그재그, 오늘의집, 크림
5천원
1만원
오프라인샵
다이소, 올리브영
온라인몰 : 대상 가맹점 공식 홈페이지 또는 모바일앱 이용 시 할인 적용
(대상 가맹점에서 각종 간편결제를 통한 결제 시 할인 제외될 수 있음)
LIFE-ZONE 내 ‘쿠팡’은 온라인 쇼핑몰(쿠팡이츠 제외) 결제 건에 한해 할인 적용
(‘쿠팡이츠’ 결제는 EAT-ZONE 할인 적용, EAT/LIFE 중복 적용 불가)
오프라인샵 : 오프라인 매장 결제 건에 한해 할인 적용 (백화점, 할인점, 공항, 기차역 등 임대 매장 제외)
PLAY ZONE
카테고리별 가맹점, 월 할인 한도 안내표
카테고리
대상 가맹점
월 할인 한도
전월실적
30만원 이상
전월실적
60만원 이상
디지털구독
넷플릭스, 유튜브 정기결제 건
5천원
1만원
쇼핑멤버십
네이버플러스 멤버십, 쿠팡 와우 멤버십
선물하기
카카오톡 선물하기
디지털구독, 쇼핑멤버십은 공식 홈페이지를 통한 정기결제(자동납부) 건에 한해 할인 적용되며, 일회성 결제 및 구글플레이/애플 앱스토어 등 앱마켓을 통한 결제(인앱결제), 통신 요금 내 합산 청구 등 가맹점 구분이 불가한 경우 할인 제외
유튜브는 유튜브 프리미엄 요금제 정기결제 건에 

### 금액 문자열 함수로 변환 함수

In [23]:
import re
import pandas as pd

def convert_korean_money(text):
    
    if pd.isna(text):
        return None
    
    text = str(text).replace(",", "").strip()
    
    # 5천원
    m = re.search(r"(\d+(?:\.\d+)?)\s*천원", text)
    if m:
        return int(float(m.group(1)) * 1000)
    
    # 1만원
    m = re.search(r"(\d+(?:\.\d+)?)\s*만원", text)
    if m:
        return int(float(m.group(1)) * 10000)
    
    # 5000원
    m = re.search(r"(\d+(?:\.\d+)?)\s*원", text)
    if m:
        return int(float(m.group(1)))
    
    return None

#### 표형 월한도 추출

In [24]:
def extract_all_monthly_limits(text):
    
    if pd.isna(text):
        return []
    
    patterns = [
        r"\d+(?:,\d+)?원",
        r"\d+(?:\.\d+)?천원",
        r"\d+(?:\.\d+)?만원"
    ]
    
    found = []
    
    for p in patterns:
        matches = re.findall(p, text)
        
        for m in matches:
            value = convert_korean_money(m)
            
            if value:
                found.append(value)
    
    # 중복 제거 + 정렬
    found = sorted(list(set(found)))
    
    return found

#### 전원 실적 추출

In [25]:
def extract_all_previous_month_spend(text):
    
    if pd.isna(text):
        return []
    
    patterns = [
        r"전월실적\s*(\d+)\s*만원\s*이상",
        r"(\d+)\s*만원\s*이상"
    ]
    
    found = []
    
    for p in patterns:
        matches = re.findall(p, text)
        
        for m in matches:
            found.append(int(m) * 10000)
    
    found = sorted(list(set(found)))
    
    return found

In [26]:
df_benefit["all_monthly_limits"] = (
    df_benefit["full_text"]
    .apply(extract_all_monthly_limits)
)

df_benefit["all_previous_month_spend"] = (
    df_benefit["full_text"]
    .apply(extract_all_previous_month_spend)
)

In [27]:
zone = df_benefit[
    df_benefit["card_name"].str.contains("BC 바로 ZONE", na=False)
]

display(zone[[
    "card_name",
    "benefit_title",
    "all_previous_month_spend",
    "all_monthly_limits",
    "full_text"
]])

,card_name,benefit_title,all_previous_month_spend,all_monthly_limits,full_text
8,[BC바로] BC 바로 ZONE 카드,EAT ZONE,"[300000, 600000]","[5000, 10000, 300000, 600000]","카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
9,[BC바로] BC 바로 ZONE 카드,LIFE ZONE,"[300000, 600000]","[5000, 10000, 300000, 600000]","카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
10,[BC바로] BC 바로 ZONE 카드,PLAY ZONE,"[300000, 600000]","[5000, 10000, 300000, 600000]","카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
11,[BC바로] BC 바로 ZONE 카드,& ZONE,"[300000, 600000]","[5000, 10000, 300000, 600000]","카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
12,[BC바로] BC 바로 ZONE 카드,해외 ZONE,[],[],"해외가맹점 및 해외 직접 구매 이용시\n전월 실적, 한도 없이 1% 결제일 할인\n..."
13,[BC바로] BC 바로 ZONE 카드,무이자할부,[50000],[50000],국내 대상 가맹점 2~3개월 상시 무이자 할부\n무이자할부 이용금액은 할인 혜택 제...
14,[BC바로] BC 바로 ZONE 카드,국제 브랜드 서비스,[],[],해외겸용 카드의 경우 Mastercard Platinum 등급 기본 서비스만 제공되...


#### 한도로 되어있는 실적 금액 제외하기

In [28]:
def remove_spend_from_limits(row):
    limits = row["all_monthly_limits"]
    spends = row["all_previous_month_spend"]
    
    if not isinstance(limits, list):
        return []
    if not isinstance(spends, list):
        spends = []
    
    return [x for x in limits if x not in spends]

df_benefit["monthly_limit_candidates"] = df_benefit.apply(
    remove_spend_from_limits,
    axis=1
)

In [29]:
zone = df_benefit[
    df_benefit["card_name"].str.contains("BC 바로 ZONE", na=False)
]

display(zone[[
    "card_name",
    "benefit_title",
    "all_previous_month_spend",
    "monthly_limit_candidates",
    "full_text"
]])

,card_name,benefit_title,all_previous_month_spend,monthly_limit_candidates,full_text
8,[BC바로] BC 바로 ZONE 카드,EAT ZONE,"[300000, 600000]","[5000, 10000]","카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
9,[BC바로] BC 바로 ZONE 카드,LIFE ZONE,"[300000, 600000]","[5000, 10000]","카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
10,[BC바로] BC 바로 ZONE 카드,PLAY ZONE,"[300000, 600000]","[5000, 10000]","카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
11,[BC바로] BC 바로 ZONE 카드,& ZONE,"[300000, 600000]","[5000, 10000]","카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
12,[BC바로] BC 바로 ZONE 카드,해외 ZONE,[],[],"해외가맹점 및 해외 직접 구매 이용시\n전월 실적, 한도 없이 1% 결제일 할인\n..."
13,[BC바로] BC 바로 ZONE 카드,무이자할부,[50000],[],국내 대상 가맹점 2~3개월 상시 무이자 할부\n무이자할부 이용금액은 할인 혜택 제...
14,[BC바로] BC 바로 ZONE 카드,국제 브랜드 서비스,[],[],해외겸용 카드의 경우 Mastercard Platinum 등급 기본 서비스만 제공되...


### 전체 카드에 반영하기

In [30]:
def map_spend_to_limit(row):
    spends = row["all_previous_month_spend"]
    limits = row["monthly_limit_candidates"]
    
    if not isinstance(spends, list):
        spends = []
    if not isinstance(limits, list):
        limits = []
    
    # 실적과 한도 개수가 같으면 순서대로 매핑
    if len(spends) == len(limits) and len(spends) > 0:
        return dict(zip(spends, limits))
    
    # 실적은 있는데 한도가 1개면 모든 실적에 같은 한도 적용
    if len(spends) > 0 and len(limits) == 1:
        return {spend: limits[0] for spend in spends}
    
    # 실적 조건 없음 + 한도만 있음
    if len(spends) == 0 and len(limits) > 0:
        return {"no_spend_condition": limits[0]}
    
    return {}

df_benefit["spend_limit_map"] = df_benefit.apply(map_spend_to_limit, axis=1)

In [31]:
display(df_benefit[[
    "card_name",
    "benefit_title",
    "all_previous_month_spend",
    "monthly_limit_candidates",
    "spend_limit_map"
]].head(30))

,card_name,benefit_title,all_previous_month_spend,monthly_limit_candidates,spend_limit_map
0,[BNK경남] 경남은행 K-패스 카드,모빌리티 할인 서비스(결제일 차감 청구),"[300000, 600000]","[6000, 8000]","{300000: 6000, 600000: 8000}"
1,[BNK경남] 경남은행 K-패스 카드,생활 할인 서비스(결제일 차감 청구),"[10000, 20000, 300000, 600000]","[1500, 6000, 8000]",{}
2,[BNK경남] 경남은행 K-패스 카드,기타 안내,[],"[8000, 10000]",{'no_spend_condition': 8000}
3,[Sh수협] 더 아우름 카드,프리미엄 바우처 (Premium Voucher),"[200000, 1000000, 2000000]",[],{}
4,[Sh수협] 더 아우름 카드,할인서비스,"[10000, 500000, 1000000, 2000000]","[5000, 15000, 45000]",{}
5,[Sh수협] 더 아우름 카드,마스터카드 월드서비스(공통),[],"[30000, 50000, 60000, 100000, 230000]",{'no_spend_condition': 30000}
6,[Sh수협] 더 아우름 카드,마스터카드 월드서비스(선택),[500000],"[20000, 25000, 30000, 34000, 44000, 50000]",{}
7,[Sh수협] 더 아우름 카드,기타 안내,[],"[190000, 200000]",{'no_spend_condition': 190000}
8,[BC바로] BC 바로 ZONE 카드,EAT ZONE,"[300000, 600000]","[5000, 10000]","{300000: 5000, 600000: 10000}"
9,[BC바로] BC 바로 ZONE 카드,LIFE ZONE,"[300000, 600000]","[5000, 10000]","{300000: 5000, 600000: 10000}"


In [32]:
df_benefit.to_csv(
    "bc_card_benefit_preprocessed_step1.csv",
    index=False,
    encoding="utf-8-sig"
)

# 할인율/적립률 보정

In [39]:
test_url = df_cards_final[
    df_cards_final["cardGdsNm"].str.contains("BC 바로 ZONE", na=False)
].iloc[0]["detail_url"]

detail_results = parse_card_detail(test_url)

for r in detail_results:
    print("=" * 80)
    print("혜택명:", r["benefit_title"])

혜택명: EAT ZONE 커피, 간식, 배달앱 7% 할인
혜택명: LIFE ZONE 온라인몰, 다이소, 올리브영 7% 할인
혜택명: PLAY ZONE 디지털 구독, 쇼핑멤버십 7% 할인
혜택명: & ZONE 전통시장 7% 할인
혜택명: 해외 ZONE 전월실적, 한도 없이 1% 할인
혜택명: 무이자할부 전국 모든 가맹점 2~3개월
혜택명: 국제 브랜드 서비스 Mastercard Platinum서비스


In [46]:
import time
import pandas as pd

all_detail_rows = []

for i, row in df_cards_final.iterrows():
    
    card_name = row["cardGdsNm"]
    detail_url = row["detail_url"]
    
    print(f"[{i+1}/{len(df_cards_final)}] 수집중: {card_name}")
    
    try:
        detail_results = parse_card_detail(detail_url)
        
        print(f"  → 혜택 {len(detail_results)}개 수집")
        
        for detail in detail_results:
            all_detail_rows.append({
                "card_name": card_name,
                "affiFirmNo": row["affiFirmNo"],
                "mbNo": row["mbNo"],
                "main_benefit": row["mainBnftCtnt"],
                "detail_url": detail_url,
                "benefit_order": detail["benefit_order"],
                "benefit_title": detail["benefit_title"],
                "sub_titles": " | ".join(detail["sub_titles"]),
                "list_texts": " | ".join(detail["list_texts"]),
                "notes": " | ".join(detail["notes"]),
                "full_text": detail["full_text"]
            })
            
    except Exception as e:
        print(f"  → 에러 발생: {e}")
    
    time.sleep(0.5)

df_detail_all = pd.DataFrame(all_detail_rows)

print("총 상세 혜택 수:", len(df_detail_all))

display(df_detail_all.head())

df_detail_all.to_csv(
    "bc_card_detail_all_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

[1/158] 수집중: [BNK경남] 경남은행 K-패스 카드
  → 혜택 3개 수집
[2/158] 수집중: [Sh수협] 더 아우름 카드
  → 혜택 5개 수집
[3/158] 수집중: [BC바로] BC 바로 ZONE 카드
  → 혜택 7개 수집
[4/158] 수집중: [BC바로] BC 바로 에어마스터
  → 혜택 5개 수집
[5/158] 수집중: [BC바로] BC 바로 에어맥스
  → 혜택 5개 수집
[6/158] 수집중: [광주] 대한항공 SKYPASS 카드(일반형)
  → 혜택 6개 수집
[7/158] 수집중: [광주] 대한항공 SKYPASS 카드(플래티늄형)
  → 혜택 7개 수집
[8/158] 수집중: [BNK경남] REXⅡ 카드
  → 혜택 4개 수집
[9/158] 수집중: [iM뱅크] iM 트래블 카드
  → 혜택 4개 수집
[10/158] 수집중: [광주] 기아챔피언스카드
  → 혜택 4개 수집
[11/158] 수집중: [Sh수협] All드림카드
  → 혜택 9개 수집
[12/158] 수집중: [BC바로] KT SUPER DC BC 바로카드
  → 혜택 3개 수집
[13/158] 수집중: [IBK기업은행] IBK포인트3.8
  → 혜택 1개 수집
[14/158] 수집중: [BC바로] BC 바로 기후동행카드
  → 혜택 7개 수집
[15/158] 수집중: [BNK부산] REXⅡ 카드
  → 혜택 5개 수집
[16/158] 수집중: [IBK기업은행] IBK포인트
  → 혜택 1개 수집
[17/158] 수집중: [BC바로] MACAO 카드
  → 혜택 4개 수집
[18/158] 수집중: [광주] 오일모아카드
  → 혜택 4개 수집
[19/158] 수집중: [광주] 에듀플러스카드
  → 혜택 4개 수집
[20/158] 수집중: [BC바로] KaPick
  → 혜택 7개 수집
[21/158] 수집중: [BC바로] KT 마이알뜰폰 BC 바로카드
  → 혜택 3개 수집
[22/158] 수집중: [BNK경남] 키스해링 신용카드
  → 혜택 1개 수집
[23/158

,card_name,affiFirmNo,mbNo,main_benefit,detail_url,benefit_order,benefit_title,sub_titles,list_texts,notes,full_text
0,[BNK경남] 경남은행 K-패스 카드,104511,039,대중교통 요금 15% 할인,https://www.bccard.com/app/card/CreditCardMain...,1,모빌리티 할인 서비스(결제일 차감 청구),대중교통 요금 15% 할인 | 친환경 모빌리티 10% 할인 | [모빌리티 할인 서비...,"대상 가맹점 : 버스(시내버스, 마을버스, 광역버스)지하철 및 GTX - 시외버스 ...",,"대중교통 요금 15% 할인\n대상 가맹점 : 버스(시내버스, 마을버스, 광역버스)지..."
1,[BNK경남] 경남은행 K-패스 카드,104511,039,대중교통 요금 15% 할인,https://www.bccard.com/app/card/CreditCardMain...,2,생활 할인 서비스(결제일 차감 청구),"이동통신요금 5% 할인 (월 1회, 최대 1,500원 할인) | 온라인 쇼핑/배달앱...","대상 가맹점 : SKT, KT, LGU+ - 이동통신요금을 본 카드로 대상 통신사에...","※ 동물병원 할인 제외 | ※ 백화점, 할인점, 쇼핑몰 등의 임대매장 제외 | ※ ...","이동통신요금 5% 할인 (월 1회, 최대 1,500원 할인)\n대상 가맹점 : SK..."
2,[BNK경남] 경남은행 K-패스 카드,104511,039,대중교통 요금 15% 할인,https://www.bccard.com/app/card/CreditCardMain...,3,기타 안내,카드발급안내 | 해외 이용 안내 | 연회비 청구 및 반환 기준 | 고객 상담 안내,해외 이용시(해외 웹사이트 포함) 이용금액에 국제브랜드사가 부과하는 수수료 (Mas...,※ BC카드 대외결제은행은 BC카드 홈페이지(고객센터-해외이용안내)에서 확인가능합니...,카드발급안내\n카드발급안내\n구분\n내용\n발급대상\n개인회원(가족회원 불가)\n브...
3,[Sh수협] 더 아우름 카드,104474,007,프리미엄 바우처 제공,https://www.bccard.com/app/card/CreditCardMain...,1,프리미엄 바우처 (Premium Voucher),바우처 4가지 중 1가지를 선택하여 이용(연1회) | 바우처 신청채널 | 바우처 신...,"BC카드 고객센터 ☏1566-7890 (평일 09:00 ~ 18:00), BC페이북...","※ 실적제외업종 : 할인받은 이용금액 전체, 장기카드대출(카드론), 단기카드대출(현...",바우처 4가지 중 1가지를 선택하여 이용(연1회)\n프리미엄 바우처 안내\nHote...
4,[Sh수협] 더 아우름 카드,104474,007,프리미엄 바우처 제공,https://www.bccard.com/app/card/CreditCardMain...,2,할인서비스,라이프스타일 패키지(옵션 패키지 중 택1) | 할인 서비스 공통기준 | 할인 제외 ...,서비스 안내 : 라이프스타일에 따른 옵션 패키지 선택 시 해당 패키지 업종 5% 결...,※ 모든 서비스에 적용되는 가맹점 및 업종은 BC카드에 등록된 가맹점 정보를 기준으...,라이프스타일 패키지(옵션 패키지 중 택1)\n서비스 안내 : 라이프스타일에 따른 옵...


#### 전처리

In [50]:
df_benefit = df_detail_all.copy()

df_benefit["raw_text"] = (
    df_benefit["benefit_title"].fillna("") + "\n" +
    df_benefit["sub_titles"].fillna("") + "\n" +
    df_benefit["list_texts"].fillna("") + "\n" +
    df_benefit["notes"].fillna("") + "\n" +
    df_benefit["full_text"].fillna("")
)

In [52]:
def classify_benefit_type(text):
    
    if pd.isna(text):
        return None
    
    text = str(text)
    
    if "마일리지" in text:
        return "마일리지"
    
    elif "캐시백" in text:
        return "캐시백"
    
    elif "적립" in text:
        return "적립"
    
    elif "할인" in text:
        return "할인"
    
    elif "바우처" in text:
        return "바우처"
    
    else:
        return "기타"


df_benefit["benefit_type"] = (
    df_benefit["raw_text"]
    .apply(classify_benefit_type)
)

In [53]:
df_benefit["benefit_rate_v2"] = df_benefit.apply(
    extract_benefit_rate_v2,
    axis=1
)

df_benefit["all_monthly_limits"] = (
    df_benefit["full_text"]
    .apply(extract_all_monthly_limits)
)

df_benefit["all_previous_month_spend"] = (
    df_benefit["full_text"]
    .apply(extract_all_previous_month_spend)
)

df_benefit["monthly_limit_candidates"] = df_benefit.apply(
    remove_spend_from_limits,
    axis=1
)

df_benefit["spend_limit_map"] = df_benefit.apply(
    map_spend_to_limit,
    axis=1
)

In [54]:
zone = df_benefit[
    df_benefit["card_name"].str.contains("BC 바로 ZONE", na=False)
]

display(zone[[
    "card_name",
    "benefit_title",
    "benefit_rate_v2",
    "all_previous_month_spend",
    "monthly_limit_candidates",
    "spend_limit_map"
]])

,card_name,benefit_title,benefit_rate_v2,all_previous_month_spend,monthly_limit_candidates,spend_limit_map
8,[BC바로] BC 바로 ZONE 카드,"EAT ZONE 커피, 간식, 배달앱 7% 할인",7.0,"[300000, 600000]","[5000, 10000]","{300000: 5000, 600000: 10000}"
9,[BC바로] BC 바로 ZONE 카드,"LIFE ZONE 온라인몰, 다이소, 올리브영 7% 할인",7.0,"[300000, 600000]","[5000, 10000]","{300000: 5000, 600000: 10000}"
10,[BC바로] BC 바로 ZONE 카드,"PLAY ZONE 디지털 구독, 쇼핑멤버십 7% 할인",7.0,"[300000, 600000]","[5000, 10000]","{300000: 5000, 600000: 10000}"
11,[BC바로] BC 바로 ZONE 카드,& ZONE 전통시장 7% 할인,7.0,"[300000, 600000]","[5000, 10000]","{300000: 5000, 600000: 10000}"
12,[BC바로] BC 바로 ZONE 카드,"해외 ZONE 전월실적, 한도 없이 1% 할인",1.0,[],[],{}
13,[BC바로] BC 바로 ZONE 카드,무이자할부 전국 모든 가맹점 2~3개월,NaN,[50000],[],{}
14,[BC바로] BC 바로 ZONE 카드,국제 브랜드 서비스 Mastercard Platinum서비스,NaN,[],[],{}


## 카드 상세혜택 139개 카드 대상

In [55]:
print("전체 상세혜택 행 수:", len(df_benefit))
print("수집 카드 수:", df_benefit["card_name"].nunique())

print("할인/적립률 추출 수:", df_benefit["benefit_rate_v2"].notna().sum())
print("전월실적 추출 수:", df_benefit["all_previous_month_spend"].apply(lambda x: len(x) > 0).sum())
print("월한도 추출 수:", df_benefit["monthly_limit_candidates"].apply(lambda x: len(x) > 0).sum())

df_benefit.to_csv(
    "bc_card_benefit_preprocessed_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

전체 상세혜택 행 수: 831
수집 카드 수: 139
할인/적립률 추출 수: 377
전월실적 추출 수: 355
월한도 추출 수: 449


In [57]:
display(df_benefit[[
    "card_name",
    "benefit_title",
    "benefit_type",
    "benefit_rate_v2",
    "all_previous_month_spend",
    "monthly_limit_candidates",
    "spend_limit_map"
]].head(10))

,card_name,benefit_title,benefit_type,benefit_rate_v2,all_previous_month_spend,monthly_limit_candidates,spend_limit_map
0,[BNK경남] 경남은행 K-패스 카드,모빌리티 할인 서비스(결제일 차감 청구),할인,15.0,"[300000, 600000]","[6000, 8000]","{300000: 6000, 600000: 8000}"
1,[BNK경남] 경남은행 K-패스 카드,생활 할인 서비스(결제일 차감 청구),할인,5.0,"[10000, 20000, 300000, 600000]","[1500, 6000, 8000]",{}
2,[BNK경남] 경남은행 K-패스 카드,기타 안내,기타,NaN,[],"[8000, 10000]",{'no_spend_condition': 8000}
3,[Sh수협] 더 아우름 카드,프리미엄 바우처 (Premium Voucher),할인,NaN,"[200000, 1000000, 2000000]",[],{}
4,[Sh수협] 더 아우름 카드,할인서비스,할인,5.0,"[10000, 500000, 1000000, 2000000]","[5000, 15000, 45000]",{}
5,[Sh수협] 더 아우름 카드,마스터카드 월드서비스(공통),할인,15.0,[],"[30000, 50000, 60000, 100000, 230000]",{'no_spend_condition': 30000}
6,[Sh수협] 더 아우름 카드,마스터카드 월드서비스(선택),할인,NaN,[500000],"[20000, 25000, 30000, 34000, 44000, 50000]",{}
7,[Sh수협] 더 아우름 카드,기타 안내,바우처,NaN,[],"[190000, 200000]",{'no_spend_condition': 190000}
8,[BC바로] BC 바로 ZONE 카드,"EAT ZONE 커피, 간식, 배달앱 7% 할인",할인,7.0,"[300000, 600000]","[5000, 10000]","{300000: 5000, 600000: 10000}"
9,[BC바로] BC 바로 ZONE 카드,"LIFE ZONE 온라인몰, 다이소, 올리브영 7% 할인",할인,7.0,"[300000, 600000]","[5000, 10000]","{300000: 5000, 600000: 10000}"


## 혜택 카테고리 분류

In [2]:
df_benefit = pd.read_csv(
    "bc_card_benefit_preprocessed_v2.csv"
)

print(df_benefit.shape)

display(df_benefit.head())

(831, 18)


,card_name,affiFirmNo,mbNo,main_benefit,detail_url,benefit_order,benefit_title,sub_titles,list_texts,notes,full_text,raw_text,benefit_type,benefit_rate_v2,all_monthly_limits,all_previous_month_spend,monthly_limit_candidates,spend_limit_map
0,[BNK경남] 경남은행 K-패스 카드,104511,39,대중교통 요금 15% 할인,https://www.bccard.com/app/card/CreditCardMain...,1,모빌리티 할인 서비스(결제일 차감 청구),대중교통 요금 15% 할인 | 친환경 모빌리티 10% 할인 | [모빌리티 할인 서비...,"대상 가맹점 : 버스(시내버스, 마을버스, 광역버스)지하철 및 GTX - 시외버스 ...",NaN,"대중교통 요금 15% 할인\n대상 가맹점 : 버스(시내버스, 마을버스, 광역버스)지...",모빌리티 할인 서비스(결제일 차감 청구)\n대중교통 요금 15% 할인 | 친환경 모...,할인,15.0,"[6000, 8000, 300000, 600000]","[300000, 600000]","[6000, 8000]","{300000: 6000, 600000: 8000}"
1,[BNK경남] 경남은행 K-패스 카드,104511,39,대중교통 요금 15% 할인,https://www.bccard.com/app/card/CreditCardMain...,2,생활 할인 서비스(결제일 차감 청구),"이동통신요금 5% 할인 (월 1회, 최대 1,500원 할인) | 온라인 쇼핑/배달앱...","대상 가맹점 : SKT, KT, LGU+ - 이동통신요금을 본 카드로 대상 통신사에...","※ 동물병원 할인 제외 | ※ 백화점, 할인점, 쇼핑몰 등의 임대매장 제외 | ※ ...","이동통신요금 5% 할인 (월 1회, 최대 1,500원 할인)\n대상 가맹점 : SK...","생활 할인 서비스(결제일 차감 청구)\n이동통신요금 5% 할인 (월 1회, 최대 1...",할인,5.0,"[1500, 6000, 8000, 10000, 20000, 300000, 600000]","[10000, 20000, 300000, 600000]","[1500, 6000, 8000]",{}
2,[BNK경남] 경남은행 K-패스 카드,104511,39,대중교통 요금 15% 할인,https://www.bccard.com/app/card/CreditCardMain...,3,기타 안내,카드발급안내 | 해외 이용 안내 | 연회비 청구 및 반환 기준 | 고객 상담 안내,해외 이용시(해외 웹사이트 포함) 이용금액에 국제브랜드사가 부과하는 수수료 (Mas...,※ BC카드 대외결제은행은 BC카드 홈페이지(고객센터-해외이용안내)에서 확인가능합니...,카드발급안내\n카드발급안내\n구분\n내용\n발급대상\n개인회원(가족회원 불가)\n브...,기타 안내\n카드발급안내 | 해외 이용 안내 | 연회비 청구 및 반환 기준 | 고객...,기타,NaN,"[8000, 10000]",[],"[8000, 10000]",{'no_spend_condition': 8000}
3,[Sh수협] 더 아우름 카드,104474,7,프리미엄 바우처 제공,https://www.bccard.com/app/card/CreditCardMain...,1,프리미엄 바우처 (Premium Voucher),바우처 4가지 중 1가지를 선택하여 이용(연1회) | 바우처 신청채널 | 바우처 신...,"BC카드 고객센터 ☏1566-7890 (평일 09:00 ~ 18:00), BC페이북...","※ 실적제외업종 : 할인받은 이용금액 전체, 장기카드대출(카드론), 단기카드대출(현...",바우처 4가지 중 1가지를 선택하여 이용(연1회)\n프리미엄 바우처 안내\nHote...,프리미엄 바우처 (Premium Voucher)\n바우처 4가지 중 1가지를 선택하...,할인,NaN,"[200000, 1000000, 2000000]","[200000, 1000000, 2000000]",[],{}
4,[Sh수협] 더 아우름 카드,104474,7,프리미엄 바우처 제공,https://www.bccard.com/app/card/CreditCardMain...,2,할인서비스,라이프스타일 패키지(옵션 패키지 중 택1) | 할인 서비스 공통기준 | 할인 제외 ...,서비스 안내 : 라이프스타일에 따른 옵션 패키지 선택 시 해당 패키지 업종 5% 결...,※ 모든 서비스에 적용되는 가맹점 및 업종은 BC카드에 등록된 가맹점 정보를 기준으...,라이프스타일 패키지(옵션 패키지 중 택1)\n서비스 안내 : 라이프스타일에 따른 옵...,할인서비스\n라이프스타일 패키지(옵션 패키지 중 택1) | 할인 서비스 공통기준 |...,할인,5.0,"[5000, 10000, 15000, 45000, 500000, 1000000, 2...","[10000, 500000, 1000000, 2000000]","[5000, 15000, 45000]",{}


In [30]:
import pandas as pd
import re

def classify_benefit_category(row):
    title = str(row.get("benefit_title", ""))
    raw = str(row.get("raw_text", ""))

    # 1순위: 혜택명 중심
    text = title.lower()

    category_rules = {
        "교통": ["대중교통", "버스", "지하철", "택시", "카카오t", "철도", "ktx", "srt", "고속버스", "하이패스", "모빌리티", "k-패스", "기후동행"],
        "카페": ["커피", "카페", "스타벅스", "투썸", "이디야", "메가커피", "컴포즈", "엔젤리너스"],
        "외식": ["외식", "음식점", "레스토랑", "패밀리 레스토랑", "아웃백", "빕스", "다이닝", "점심"],
        "배달": ["배달", "배달앱", "배달의민족", "배민", "요기요", "쿠팡이츠"],
        "마트·편의점": ["마트", "대형마트", "대형할인점", "이마트", "트레이더스", "롯데마트", "홈플러스", "편의점", "gs25", "cu", "세븐일레븐", "이마트24", "장보기", "전통시장"],
        "쇼핑": ["쇼핑", "온라인쇼핑", "온라인몰", "오프라인 쇼핑", "쿠팡", "옥션", "g마켓", "11번가", "ssg", "신세계백화점", "백화점", "다이소", "올리브영", "무신사", "w concept", "소셜커머스"],
        "통신": ["통신", "이동통신", "휴대폰", "skt", "kt", "lg u+", "알뜰폰", "단말기"],
        "구독": ["구독", "ott", "스트리밍", "넷플릭스", "유튜브", "티빙", "웨이브", "디즈니", "멜론", "쇼핑멤버십"],
        "주유·충전": ["주유", "충전소", "lpg", "전기차", "수소차", "gs칼텍스", "sk주유소", "sk에너지", "s-oil", "현대오일뱅크"],
        "여행·항공": ["항공", "대한항공", "아시아나", "스카이패스", "마일리지", "여행", "호텔", "숙박", "공항", "라운지", "발레파킹", "위탁수하물", "해외", "환율", "해외atm", "해외 인출", "해외 가맹점", "해외이용"],
        "의료": ["병원", "의원", "한의원", "약국", "의료", "동물병원"],
        "교육": ["교육", "학원", "온라인교육", "ebs", "교재", "서점", "도서", "아카데미"],
        "문화·여가": ["영화", "cgv", "롯데시네마", "메가박스", "공연", "문화", "놀이공원", "레저", "골프", "경기장", "로스트아크", "게임"],
        "반려동물": ["반려동물", "동물병원", "펫", "애견"],
        "자동차": ["자동차", "정비", "하이패스"],
        "아파트·공과금": ["아파트관리비", "관리비", "공과금"],
        "포인트": ["금융", "포인트", "캐시백", "에코머니", "top포인트", "l.point", "oh! point", "paybook머니", "페이북머니", "ok cashbag"],
        "간편결제": ["간편결제", "카카오페이", "네이버페이", "ssgpay", "페이", "pay", "qr"],
        "기본혜택": ["전가맹점", "모든 가맹점", "국내외 가맹점", "기본 할인", "기본 적립", "기본캐시백", "all"]
    }

    matched = []

    for category, keywords in category_rules.items():
            for keyword in sorted(keywords, key=len, reverse=True):
                if keyword.lower() in text:
                    matched.append(category)
                    break

    # 혜택명에서 못 잡힌 경우에만 raw_text로 보조 분류
    if not matched:
        text = raw.lower()
        for category, keywords in category_rules.items():
            for keyword in sorted(keywords, key=len, reverse=True):
                if keyword.lower() in text:
                    matched.append(category)
                    break

    if not matched:
        return "기타"

    matched = list(set(matched))
    
    # =========================
    # 충돌 해결 규칙
    # =========================
    
    # 쿠팡이츠 같은 경우
    if "배달" in matched and "쇼핑" in matched:
        
        delivery_keywords = [
            "쿠팡이츠",
            "배달의민족",
            "배민",
            "요기요"
        ]
        
        if any(keyword.lower() in text for keyword in delivery_keywords):
            matched.remove("쇼핑")
    
    # 네이버페이 같은 경우
    if "간편결제" in matched and "쇼핑" in matched:
        
        pay_keywords = [
            "네이버페이",
            "카카오페이",
            "pay"
        ]
        
        if any(keyword.lower() in text for keyword in pay_keywords):
            matched.remove("쇼핑")
    
    return " / ".join(sorted(matched))


df_benefit["benefit_category_final"] = df_benefit.apply(
    classify_benefit_category,
    axis=1
)

display(df_benefit[[
    "card_name",
    "benefit_title",
    "benefit_category_final"
]].head(50))

df_benefit["benefit_category_final"].value_counts()

,card_name,benefit_title,benefit_category_final
0,[BNK경남] 경남은행 K-패스 카드,모빌리티 할인 서비스(결제일 차감 청구),교통
1,[BNK경남] 경남은행 K-패스 카드,생활 할인 서비스(결제일 차감 청구),간편결제 / 구독 / 마트·편의점 / 반려동물 / 배달 / 여행·항공 / 의료 / ...
3,[Sh수협] 더 아우름 카드,프리미엄 바우처 (Premium Voucher),간편결제 / 마트·편의점 / 문화·여가 / 쇼핑 / 아파트·공과금 / 여행·항공 /...
4,[Sh수협] 더 아우름 카드,할인서비스,간편결제 / 교육 / 마트·편의점 / 문화·여가 / 반려동물 / 아파트·공과금 / ...
8,[BC바로] BC 바로 ZONE 카드,"EAT ZONE 커피, 간식, 배달앱 7% 할인",배달 / 카페
9,[BC바로] BC 바로 ZONE 카드,"LIFE ZONE 온라인몰, 다이소, 올리브영 7% 할인",쇼핑
10,[BC바로] BC 바로 ZONE 카드,"PLAY ZONE 디지털 구독, 쇼핑멤버십 7% 할인",구독 / 쇼핑
11,[BC바로] BC 바로 ZONE 카드,& ZONE 전통시장 7% 할인,마트·편의점
12,[BC바로] BC 바로 ZONE 카드,"해외 ZONE 전월실적, 한도 없이 1% 할인",여행·항공
15,[BC바로] BC 바로 에어마스터,"기본 적립 1,500원당 1 스카이패스 마일리지",기본혜택 / 여행·항공


benefit_category_final
포인트                                                    69
쇼핑                                                     53
여행·항공                                                  53
문화·여가                                                  52
교통                                                     43
                                                       ..
마트·편의점 / 쇼핑 / 외식 / 주유·충전 / 카페                           1
문화·여가 / 쇼핑 / 아파트·공과금 / 여행·항공 / 의료 / 주유·충전 / 카페 / 통신     1
교통 / 마트·편의점 / 쇼핑 / 여행·항공 / 외식 / 카페 / 통신                 1
간편결제 / 교통 / 자동차 / 통신                                    1
교육 / 마트·편의점 / 쇼핑                                        1
Name: count, Length: 156, dtype: int64

In [35]:
# 제거할 불필요 혜택명 키워드
remove_keywords = [
    "기타 안내",
    "국제 브랜드 서비스",
    "유의사항",
    "카드 이용 안내",
    "연회비 안내",
    "무이자 할부",
    "기타안내",
    "마스터카드",
    "마스터 브랜드",
    "VISA 서비스",
    "Priority Pass",
    "공통 서비스"
]

# 제거 함수
def is_unnecessary_benefit(title):
    
    if pd.isna(title):
        return False
    
    title = str(title)
    
    for keyword in remove_keywords:
        if keyword.lower() in title.lower():
            return True
    
    return False

# 제거 전 확인
print("제거 전 행 수:", len(df_benefit))

# 불필요 혜택 제거
df_benefit = df_benefit[
    ~df_benefit["benefit_title"].apply(is_unnecessary_benefit)
].copy()

# 제거 후 확인
print("제거 후 행 수:", len(df_benefit))

# 남은 혜택 확인
display(df_benefit[[
    "card_name",
    "benefit_title",
    "benefit_category_final"
]].head(60))

제거 전 행 수: 689
제거 후 행 수: 689


,card_name,benefit_title,benefit_category_final
0,[BNK경남] 경남은행 K-패스 카드,모빌리티 할인 서비스(결제일 차감 청구),교통
1,[BNK경남] 경남은행 K-패스 카드,생활 할인 서비스(결제일 차감 청구),간편결제 / 구독 / 마트·편의점 / 반려동물 / 배달 / 여행·항공 / 의료 / ...
3,[Sh수협] 더 아우름 카드,프리미엄 바우처 (Premium Voucher),간편결제 / 마트·편의점 / 문화·여가 / 쇼핑 / 아파트·공과금 / 여행·항공 /...
4,[Sh수협] 더 아우름 카드,할인서비스,간편결제 / 교육 / 마트·편의점 / 문화·여가 / 반려동물 / 아파트·공과금 / ...
8,[BC바로] BC 바로 ZONE 카드,"EAT ZONE 커피, 간식, 배달앱 7% 할인",배달 / 카페
9,[BC바로] BC 바로 ZONE 카드,"LIFE ZONE 온라인몰, 다이소, 올리브영 7% 할인",쇼핑
10,[BC바로] BC 바로 ZONE 카드,"PLAY ZONE 디지털 구독, 쇼핑멤버십 7% 할인",구독 / 쇼핑
11,[BC바로] BC 바로 ZONE 카드,& ZONE 전통시장 7% 할인,마트·편의점
12,[BC바로] BC 바로 ZONE 카드,"해외 ZONE 전월실적, 한도 없이 1% 할인",여행·항공
15,[BC바로] BC 바로 에어마스터,"기본 적립 1,500원당 1 스카이패스 마일리지",기본혜택 / 여행·항공


In [36]:
final_cols = [
    "card_name",
    "benefit_title",
    "benefit_category_final",
    "benefit_type",
    "benefit_rate_v2",
    "all_previous_month_spend",
    "monthly_limit_candidates",
    "spend_limit_map",
    "main_benefit",
    "detail_url",
    "raw_text",
]

if "full_text" in df_benefit.columns:
    final_cols.append("full_text")

final_cols = [c for c in final_cols if c in df_benefit.columns]

df_benefit_final = df_benefit[final_cols].copy()

df_benefit_final.to_csv(
    "bc_card_benefit_final_before_recommendation.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df_benefit_final.shape)
display(df_benefit_final.head())

(689, 12)


,card_name,benefit_title,benefit_category_final,benefit_type,benefit_rate_v2,all_previous_month_spend,monthly_limit_candidates,spend_limit_map,main_benefit,detail_url,raw_text,full_text
0,[BNK경남] 경남은행 K-패스 카드,모빌리티 할인 서비스(결제일 차감 청구),교통,할인,15.0,"[300000, 600000]","[6000, 8000]","{300000: 6000, 600000: 8000}",대중교통 요금 15% 할인,https://www.bccard.com/app/card/CreditCardMain...,모빌리티 할인 서비스(결제일 차감 청구)\n대중교통 요금 15% 할인 | 친환경 모...,"대중교통 요금 15% 할인\n대상 가맹점 : 버스(시내버스, 마을버스, 광역버스)지..."
1,[BNK경남] 경남은행 K-패스 카드,생활 할인 서비스(결제일 차감 청구),간편결제 / 구독 / 마트·편의점 / 반려동물 / 배달 / 여행·항공 / 의료 / ...,할인,5.0,"[10000, 20000, 300000, 600000]","[1500, 6000, 8000]",{},대중교통 요금 15% 할인,https://www.bccard.com/app/card/CreditCardMain...,"생활 할인 서비스(결제일 차감 청구)\n이동통신요금 5% 할인 (월 1회, 최대 1...","이동통신요금 5% 할인 (월 1회, 최대 1,500원 할인)\n대상 가맹점 : SK..."
3,[Sh수협] 더 아우름 카드,프리미엄 바우처 (Premium Voucher),간편결제 / 마트·편의점 / 문화·여가 / 쇼핑 / 아파트·공과금 / 여행·항공 /...,할인,NaN,"[200000, 1000000, 2000000]",[],{},프리미엄 바우처 제공,https://www.bccard.com/app/card/CreditCardMain...,프리미엄 바우처 (Premium Voucher)\n바우처 4가지 중 1가지를 선택하...,바우처 4가지 중 1가지를 선택하여 이용(연1회)\n프리미엄 바우처 안내\nHote...
4,[Sh수협] 더 아우름 카드,할인서비스,간편결제 / 교육 / 마트·편의점 / 문화·여가 / 반려동물 / 아파트·공과금 / ...,할인,5.0,"[10000, 500000, 1000000, 2000000]","[5000, 15000, 45000]",{},프리미엄 바우처 제공,https://www.bccard.com/app/card/CreditCardMain...,할인서비스\n라이프스타일 패키지(옵션 패키지 중 택1) | 할인 서비스 공통기준 |...,라이프스타일 패키지(옵션 패키지 중 택1)\n서비스 안내 : 라이프스타일에 따른 옵...
8,[BC바로] BC 바로 ZONE 카드,"EAT ZONE 커피, 간식, 배달앱 7% 할인",배달 / 카페,할인,7.0,"[300000, 600000]","[5000, 10000]","{300000: 5000, 600000: 10000}",온라인몰/커피/배달 7% 할인,https://www.bccard.com/app/card/CreditCardMain...,"EAT ZONE 커피, 간식, 배달앱 7% 할인\n\n커피, 간식 : 현장결제 또는...","카테고리별 가맹점, 월 할인 한도 안내표\n카테고리\n대상 가맹점\n월 할인 한도\..."
